# Random Walk Fine-Tuning

Fine-tunes the Random Walk model (fixed probability of switching to a random different role)
on both aggregate and switch-stage metrics.

In [1]:
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from itertools import product
from scipy.optimize import minimize

# Shared package
from shared import EXPORTS_DIR, DATA_ROOT
from shared.constants import (
    F, T, M,
    ROLE_NAMES, ROLE_SHORT, ROLE_CHAR_TO_IDX, GAME_ROLE_TO_IDX,
    ALL_ROLE_COMBOS, EPSILON, MAX_STAGES, TURNS_PER_STAGE,
)
from shared.parsing import canonical_combo, get_canonical_combos
from shared.inference import (
    utility_based_prior, uniform_prior,
    bayesian_update, action_prob, preferred_action, game_step,
    softmax_role_dist, combo_marginal,
    ATTACK, DEFEND, HEAL,
)
from shared.evaluation import (
    run_predictions, compute_pearson, compute_log_likelihood, extract_metrics,
)

# Still need online_model_sim for load_team_rounds
OMS_DIR = Path(DATA_ROOT).parent.parent / 'computational_model' / 'analysis'
sys.path.insert(0, str(OMS_DIR))
import online_model_sim as oms

## 1. Load Data

In [2]:
# Monkey-patch env paths to use shared data directory
oms.VALUE_MATRICES_DIR = DATA_ROOT / 'human_envs_value_matrices'
oms.ENVS_DIR = DATA_ROOT / 'envs'

# Monkey-patch config loader to avoid JAX dependency
from shared.env_loading import load_env_config
import re as _re

def _load_config_no_jax(config_path):
    text = Path(config_path).read_text()
    team_max_hp = int(_re.search(r'TEAM_MAX_HP\s*=\s*(\d+)', text).group(1))
    enemy_max_hp = int(_re.search(r'ENEMY_MAX_HP\s*=\s*(\d+)', text).group(1))
    boss_damage = float(_re.search(r'BOSS_DAMAGE\s*=\s*([\d.]+)', text).group(1))
    ps_match = _re.search(r'PLAYER_STATS\s*=\s*(?:jnp\.array|np\.array)?\(?\s*(\[\[.+?\]\])\s*\)?', text, _re.DOTALL)
    rows = _re.findall(r'\[([^\[\]]+)\]', ps_match.group(1))
    player_stats = np.array([[float(x) for x in row.split(',')] for row in rows])
    class Config: pass
    cfg = Config()
    cfg.TEAM_MAX_HP = team_max_hp
    cfg.ENEMY_MAX_HP = enemy_max_hp
    cfg.BOSS_DAMAGE = boss_damage
    cfg.PLAYER_STATS = player_stats
    return cfg

oms.load_config_module = _load_config_no_jax

# Load data
DATA_DIRS = [
    str(EXPORTS_DIR / 'bayesian-role-specialization-2026-03-06-09-54-19'),
    str(EXPORTS_DIR / 'bayesian-role-specialization-2026-03-18-15-47-09'),
]

records = oms.load_team_rounds(data_dirs=DATA_DIRS)
n_envs = len(set(r['env_id'] for r in records))
print(f'Loaded {len(records)} team-rounds across {n_envs} environments')

Loaded 66 team-rounds across 6 environments


## 2. Aggregate Fine-Tuning

In [3]:
metric = 'combo_r'

def pick_best(results, metric='combo_r'):
    valid = [r for r in results if not np.isnan(r.get(metric, float('nan')))]
    if not valid:
        return results[0]
    return max(valid, key=lambda r: r[metric])

def evaluate_rw(records, eps):
    """Run Random Walk and return fit metrics."""
    def predict_fn(record):
        preds = []
        for s, human_combo in enumerate(record['stage_roles']):
            prev = record['stage_roles'][s - 1] if s > 0 else None
            if prev is None:
                dist = {c: 1.0 / 27 for c in ALL_ROLE_COMBOS}
            else:
                dist = {}
                for combo in ALL_ROLE_COMBOS:
                    p = 1.0
                    for i, (c, prev_c) in enumerate(zip(combo, prev)):
                        p *= (1.0 - eps) if c == prev_c else (eps / 2.0)
                    dist[combo] = p
                total = sum(dist.values())
                dist = {c: p / total for c, p in dist.items()}
            marg = np.zeros(3)
            for combo, prob in dist.items():
                for c in combo:
                    marg[ROLE_CHAR_TO_IDX[c]] += prob
            marg /= 3.0
            preds.append({'predicted_dist': dist, 'human_combo': human_combo, 'model_marginal': marg})
        return preds
    results = run_predictions(records, predict_fn)
    corrs = compute_pearson(results)
    ll = compute_log_likelihood(results)
    g = corrs.get('__global__', {})
    return {
        'eps': eps,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }

# Sweep eps from 0.01 to 0.99
rw_eps_vals = np.linspace(0.01, 0.99, 50)
rw_sweep = []
for eps in rw_eps_vals:
    rw_sweep.append(evaluate_rw(records, eps))

best_rw = pick_best(rw_sweep, metric)

# Scipy polish
def objective_rw(params):
    return -evaluate_rw(records, params[0])[metric]

rw_opt = minimize(objective_rw, [best_rw['eps']], method='L-BFGS-B',
                  bounds=[(0.001, 0.999)], options={'maxiter': 50, 'ftol': 1e-6})
rw_opt_eval = evaluate_rw(records, rw_opt.x[0])
rw_sweep.append(rw_opt_eval)
best_rw = pick_best(rw_sweep, metric)

print(f'Best Random Walk: eps={best_rw["eps"]:.4f}')
print(f'  combo_r={best_rw["combo_r"]:.4f}  marg_r={best_rw["marg_r"]:.4f}  mean_ll={best_rw["mean_ll"]:.4f}')

Best Random Walk: eps=0.2570
  combo_r=0.4510  marg_r=0.5275  mean_ll=-2.8763


## 3. Switch-Stage Fine-Tuning

In [4]:
def filter_switch_stages(all_results):
    """Keep only predictions where the human combo changed from the previous stage."""
    filtered = {}
    for env_id, data in all_results.items():
        new_team_preds = []
        for team_preds in data['team_predictions']:
            filtered_preds = []
            for s, pred in enumerate(team_preds):
                if s > 0 and pred['human_combo'] != team_preds[s - 1]['human_combo']:
                    filtered_preds.append(pred)
            new_team_preds.append(filtered_preds)

        canon_combos = data['canonical_combos']
        stat_profile = data['stat_profile']
        stage_predicted = defaultdict(lambda: defaultdict(float))
        stage_human = defaultdict(lambda: defaultdict(int))
        stage_model_marg = defaultdict(lambda: np.zeros(3))
        stage_human_marg = defaultdict(lambda: np.zeros(3))
        stage_counts = defaultdict(int)
        max_stages = 0

        for team_preds in new_team_preds:
            for s, pred in enumerate(team_preds):
                stage_counts[s] += 1
                max_stages = max(max_stages, s + 1)
                for combo, prob in pred['predicted_dist'].items():
                    stage_predicted[s][canonical_combo(combo, stat_profile)] += prob
                stage_human[s][canonical_combo(pred['human_combo'], stat_profile)] += 1
                stage_model_marg[s] += pred['model_marginal']
                stage_human_marg[s] += combo_marginal(pred['human_combo'])

        if max_stages == 0:
            continue

        predicted_avg, model_marg_avg, human_marg_avg = {}, {}, {}
        for s in range(max_stages):
            n = stage_counts[s]
            if n > 0:
                predicted_avg[s] = {cc: stage_predicted[s].get(cc, 0.0) / n for cc in canon_combos}
                model_marg_avg[s] = stage_model_marg[s] / n
                human_marg_avg[s] = stage_human_marg[s] / n

        filtered[env_id] = dict(data)
        filtered[env_id].update({
            'max_stages': max_stages,
            'stage_predicted': predicted_avg,
            'stage_human': dict(stage_human),
            'stage_counts': dict(stage_counts),
            'team_predictions': new_team_preds,
            'stage_model_marginal': model_marg_avg,
            'stage_human_marginal': human_marg_avg,
        })

    return filtered

In [5]:
def evaluate_rw_switch(records, eps):
    """Run Random Walk on full history, compute metrics only on switch stages."""
    def predict_fn(record):
        preds = []
        for s, human_combo in enumerate(record['stage_roles']):
            prev = record['stage_roles'][s - 1] if s > 0 else None
            if prev is None:
                dist = {c: 1.0 / 27 for c in ALL_ROLE_COMBOS}
            else:
                dist = {}
                for combo in ALL_ROLE_COMBOS:
                    p = 1.0
                    for i, (c, prev_c) in enumerate(zip(combo, prev)):
                        p *= (1.0 - eps) if c == prev_c else (eps / 2.0)
                    dist[combo] = p
                total = sum(dist.values())
                dist = {c: p / total for c, p in dist.items()}
            marg = np.zeros(3)
            for combo, prob in dist.items():
                for c in combo:
                    marg[ROLE_CHAR_TO_IDX[c]] += prob
            marg /= 3.0
            preds.append({'predicted_dist': dist, 'human_combo': human_combo, 'model_marginal': marg})
        return preds
    full_results = run_predictions(records, predict_fn)
    sw_results = filter_switch_stages(full_results)
    if not sw_results:
        return {'eps': eps, 'combo_r': float('nan'), 'marg_r': float('nan'), 'mean_ll': float('nan')}
    corrs = compute_pearson(sw_results)
    ll = compute_log_likelihood(sw_results)
    g = corrs.get('__global__', {})
    return {
        'eps': eps,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }

# Sweep + scipy polish
rw_sw_sweep = [evaluate_rw_switch(records, e) for e in np.linspace(0.01, 0.99, 50)]
best_rw_sw = pick_best(rw_sw_sweep, metric)

def objective_rw_sw(params):
    return -evaluate_rw_switch(records, params[0])[metric]

rw_sw_opt = minimize(objective_rw_sw, [best_rw_sw['eps']], method='L-BFGS-B',
                     bounds=[(0.001, 0.999)], options={'maxiter': 50, 'ftol': 1e-6})
rw_sw_sweep.append(evaluate_rw_switch(records, rw_sw_opt.x[0]))
best_rw_sw = pick_best(rw_sw_sweep, metric)

print(f'Best Random Walk (switch-stage): eps={best_rw_sw["eps"]:.4f}')
print(f'  combo_r={best_rw_sw["combo_r"]:.4f}  marg_r={best_rw_sw["marg_r"]:.4f}')

Best Random Walk (switch-stage): eps=0.4958
  combo_r=0.2053  marg_r=0.3588


## 4. Save Parameters

In [6]:
output = {
    'metric_optimized': metric,
    'aggregate_tuned': {
        'eps': best_rw['eps'],
        'combo_r': best_rw['combo_r'], 'marg_r': best_rw['marg_r'], 'mean_ll': best_rw['mean_ll'],
    },
    'switch_stage_tuned': {
        'eps': best_rw_sw['eps'],
        'combo_r': best_rw_sw['combo_r'], 'marg_r': best_rw_sw['marg_r'], 'mean_ll': best_rw_sw['mean_ll'],
    },
}

with open('random_walk_params.json', 'w') as f:
    json.dump(output, f, indent=2)
print('Saved to random_walk_params.json')
print(json.dumps(output, indent=2))

Saved to random_walk_params.json
{
  "metric_optimized": "combo_r",
  "aggregate_tuned": {
    "eps": 0.2569688157813117,
    "combo_r": 0.45104531723005137,
    "marg_r": 0.5274508153914093,
    "mean_ll": -2.8763063844328056
  },
  "switch_stage_tuned": {
    "eps": 0.49583730297582795,
    "combo_r": 0.20525681019583256,
    "marg_r": 0.35881447385566356,
    "mean_ll": -3.1889369999694375
  }
}
